# Session 6: PyTorch & Transformers

**LLM Engineering Masterclass -- GrowAI**

---

## What We Are Building Today

1. Create and manipulate PyTorch tensors
2. Build a simple neural network with nn.Module
3. Write a training loop and watch the model learn
4. Understand the full Transformer architecture (theory)

## Setup

Install PyTorch. If you are on Google Colab, PyTorch is pre-installed.

In [ ]:
!pip install -q torch

---

## Topic 1: PyTorch Fundamentals

PyTorch is the library that builds neural networks. Every major LLM (Llama, Mistral, GPT) was built with PyTorch.

The most important concept: **tensors** -- containers for numbers with GPU and autograd superpowers.

### Step 1: Import PyTorch

In [ ]:
import torch
import torch.nn as nn

print(f"PyTorch version: {torch.__version__}")

### Step 2: Creating Tensors

A tensor is like a Python list, but designed for math operations and can run on GPU.

In [ ]:
# 1D tensor (a list of numbers)
numbers = torch.tensor([1, 2, 3, 4, 5])
print("1D tensor:", numbers)
print("Shape:", numbers.shape)
print("Type:", type(numbers))

In [ ]:
# 2D tensor (a table -- rows and columns)
table = torch.tensor([[1, 2, 3],
                       [4, 5, 6]])
print("2D tensor:\n", table)
print("Shape:", table.shape)

In [ ]:
# Useful tensor creation shortcuts
zeros = torch.zeros(3, 4)   # 3 rows, 4 columns, all zeros
ones = torch.ones(2, 2)     # 2 rows, 2 columns, all ones
random = torch.randn(2, 3)  # 2 rows, 3 columns, random numbers

print("Zeros:\n", zeros)
print("\nOnes:\n", ones)
print("\nRandom:\n", random)

# Notice the 0. and 1. -- the dot means floating-point (decimal)
# PyTorch defaults to floats because neural network math needs decimals

In [ ]:
# Basic math on tensors
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])

print("Add:", a + b)
print("Multiply:", a * b)
print("Mean:", a.mean())

### Step 3: nn.Module -- Building a Neural Network

Every neural network in PyTorch is built using `nn.Module`. Two parts:
- `__init__` -- what layers does the network have?
- `forward` -- how does data flow through?

In [ ]:
class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer = nn.Linear(1, 1)

    def forward(self, x):
        return self.layer(x)

In [ ]:
# Create the model
model = SimpleModel()
print(model)

# named_parameters() returns a generator -- gives items one at a time
# Use a for loop to see the actual weights
print("\nModel weights (random starting values):")
for name, param in model.named_parameters():
    print(f"  {name}: {param.data}")

---

## Topic 2: Training Loop

The training loop is the heart of deep learning. Four steps, repeated hundreds of times:

1. **Forward pass** -- data goes in, prediction comes out
2. **Loss** -- compare prediction to correct answer (how wrong?)
3. **Backward pass** -- figure out which weights caused the error
4. **Update** -- adjust those weights slightly

Analogy: archery practice -- shoot, measure, adjust, repeat.

In [ ]:
# The problem: teach the model that y = 2x
# Training data -- input and correct output pairs
x_train = torch.tensor([[1.0], [2.0], [3.0], [4.0], [5.0]])
y_train = torch.tensor([[2.0], [4.0], [6.0], [8.0], [10.0]])

print("Training data:")
for i in range(len(x_train)):
    print(f"  x={x_train[i].item():.0f} -> y={y_train[i].item():.0f}")

In [ ]:
# Three things needed for training:
model = SimpleModel()                                  # Fresh model (random weights)
loss_fn = nn.MSELoss()                                 # How to measure error
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)  # How to adjust weights

# lr = learning rate (how big each adjustment is)

In [ ]:
# The Training Loop
for epoch in range(500):
    # 1. Forward pass
    predictions = model(x_train)

    # 2. Calculate loss
    loss = loss_fn(predictions, y_train)

    # 3. Backward pass
    loss.backward()

    # 4. Update weights
    optimizer.step()

    # 5. Clear gradients
    optimizer.zero_grad()

    # Print every 50 rounds
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1:3d}, Loss: {loss.item():.4f}")

In [ ]:
# Test: what does the model predict for x = 6?
# (it never saw 6 during training!)
test_input = torch.tensor([[6.0]])
prediction = model(test_input)
print(f"Input: 6.0, Predicted: {prediction.item():.2f}, Expected: 12.0")

test_input_2 = torch.tensor([[10.0]])
prediction_2 = model(test_input_2)
print(f"Input: 10.0, Predicted: {prediction_2.item():.2f}, Expected: 20.0")

In [ ]:
# What did the model learn?
print("Learned weights:")
for name, param in model.named_parameters():
    print(f"  {name}: {param.data}")

# Weight should be close to 2.0, bias close to 0.0
# Because the rule is y = 2x + 0

### Experiment: Try Different Learning Rates

What happens when the learning rate is too high or too low?

In [ ]:
# Try lr=0.1 (too high?)
model_fast = SimpleModel()
optimizer_fast = torch.optim.SGD(model_fast.parameters(), lr=0.1)

print("--- Learning Rate = 0.1 ---")
for epoch in range(500):
    pred = model_fast(x_train)
    loss = loss_fn(pred, y_train)
    loss.backward()
    optimizer_fast.step()
    optimizer_fast.zero_grad()
    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch+1:3d}, Loss: {loss.item():.6f}")

# Try lr=0.001 (too low?)
model_slow = SimpleModel()
optimizer_slow = torch.optim.SGD(model_slow.parameters(), lr=0.001)

print("\n--- Learning Rate = 0.001 ---")
for epoch in range(500):
    pred = model_slow(x_train)
    loss = loss_fn(pred, y_train)
    loss.backward()
    optimizer_slow.step()
    optimizer_slow.zero_grad()
    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch+1:3d}, Loss: {loss.item():.6f}")

---

## Topic 3: Transformer Architecture (Theory)

The Transformer is the specific neural network architecture behind every LLM.

### Key Components

| Component | What It Does |
|-----------|-------------|
| Self-Attention | For every word, decides which other words to focus on |
| Multi-Head Attention | Multiple attention operations in parallel (different perspectives) |
| Positional Encoding | Tells the model word ORDER (without it, "dog bites man" = "man bites dog") |
| Feed-Forward Network | Transforms each word's representation after attention |
| Residual Connections | Safety net -- original input is never lost |
| Layer Normalization | Keeps numbers in a stable range during training |

### Decoder-Only Architecture

Every LLM (ChatGPT, Llama, Gemini) is **decoder-only**. It predicts the next token based on everything before it. One token at a time.

See the whiteboard diagrams and teaching notes for detailed visual explanations.

---

## Session Summary

| Topic | Key Takeaway |
|-------|-------------|
| PyTorch Tensors | Containers for numbers with GPU + autograd superpowers |
| nn.Module | Blueprint for neural networks: `__init__` (layers) + `forward` (data flow) |
| Training Loop | Forward -> Loss -> Backward -> Update -> Repeat. Loss goes down. Model learns. |
| The Model IS the Weights | A 4GB model file is just billions of carefully tuned numbers |
| Transformer | Multi-head attention + positional encoding + decoder-only = every LLM |

**Next session:** Tokenizers, embeddings, HuggingFace, and the open-source LLM landscape.